# scRNA-seq: Dimensionality Reduction and Clustering

**Tier 3 — Applied Bioinformatics | Module 30 · Notebook 2**

*Prerequisites: Notebook 1 (QC & Preprocessing)*

---

**By the end of this notebook you will be able to:**
1. Apply PCA to reduce dimensionality and select the number of significant PCs
2. Build a k-nearest-neighbor graph and compute UMAP / t-SNE embeddings
3. Cluster cells with the Leiden algorithm and interpret resolution parameters
4. Identify differentially expressed genes between clusters (Wilcoxon, t-test)
5. Visualize cluster marker genes as dot plots, violin plots, and feature plots


**Key resources:**
- [Scanpy clustering tutorial](https://scanpy-tutorials.readthedocs.io/en/latest/pbmc3k.html)
- [UMAP documentation](https://umap-learn.readthedocs.io/)
- [Understanding UMAP (Coenen & Pearce)](https://pair-code.github.io/understanding-umap/)

---[< Previous: scRNA-seq: Quality Control and Preprocessing](01_scrna_preprocessing.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: scRNA-seq: Cell Type Annotation >](03_cell_type_annotation.ipynb)---

## Why dimensionality reduction matters

A 2,000-gene HVG matrix is too high-dimensional to cluster or visualize directly. PCA first compresses this into ~30–50 meaningful axes that capture most biological variance while discarding technical noise. The k-NN graph built on PCA coordinates encodes the manifold structure of the data — which cells are similar to which — and both UMAP embeddings and graph-based clustering (Leiden) operate on this graph. Understanding the graph is the key to understanding why changing `n_neighbors` or `resolution` changes your results.

In [ ]:
# Setup: recreate the preprocessed AnnData from Notebook 1
# (or load from disk if running after Notebook 1)
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

try:
    adata = ad.read_h5ad('/tmp/scrna_preprocessed.h5ad')
    print(f"Loaded from disk: {adata.shape}")
except FileNotFoundError:
    # Rebuild synthetic dataset
    rng = np.random.default_rng(42)
    n_cells, n_genes = 500, 1500
    cell_types = ["T_cell", "B_cell", "Monocyte", "NK_cell", "Dendritic"]
    cells_per = n_cells // len(cell_types)
    labels = np.repeat(cell_types, cells_per)
    
    counts = rng.negative_binomial(2, 0.9, (n_cells, n_genes)).astype(np.float32)
    type_idx = {ct: np.where(labels == ct)[0] for ct in cell_types}
    offsets = [50, 200, 400, 700, 1100]
    for ct, off in zip(cell_types, offsets):
        counts[np.ix_(type_idx[ct], [off, off+1, off+2])] += rng.poisson(20, (cells_per, 3))
    
    # Normalize and log-transform
    cell_totals = counts.sum(axis=1, keepdims=True)
    X_norm = np.log1p(counts / cell_totals * 1e4)
    
    adata = ad.AnnData(
        X=X_norm,
        obs=pd.DataFrame({'cell_type': labels}, index=[f'CELL_{i:04d}' for i in range(n_cells)]),
        var=pd.DataFrame({'hvg': True}, index=[f'GENE{i:04d}' for i in range(n_genes)])
    )
    print(f"Created synthetic dataset: {adata.shape}")


## 1. Principal Component Analysis

### Why scale before PCA?
Log-normalized expression values still vary in magnitude across genes — a housekeeping gene may have mean expression of 3 while a rare marker has mean 0.1. PCA would be dominated by high-mean genes regardless of their biological relevance. Scaling standardizes each gene to zero mean and unit variance (z-score), so every gene contributes equally to the principal components.

**`max_value=10` clipping**: extreme outlier cells (often doublets that survived QC) can produce single-gene z-scores of 50+, distorting PCs. Clipping at 10 standard deviations prevents this.

### Choosing the number of PCs
The elbow plot shows cumulative variance explained by each PC. The "elbow" — where each additional PC adds diminishing variance — suggests how many PCs capture biological signal vs noise. For PBMCs: typically 10–20 PCs. For complex tissue samples: 30–50 PCs.

**Rule of thumb**: Use 2× the PC at the elbow. Over-including PCs adds noise but rarely destroys structure; under-including can merge distinct cell types.

### PC loadings interpretation
Each PC is a weighted combination of genes. PC1 often corresponds to the dominant cell type axis (e.g., immune vs stromal). Inspect `sc.pl.pca_loadings(adata, components=[1,2,3])` to see which genes drive each component.

In [ ]:
# PCA on the HVG matrix
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Step 1: Scale genes (center + unit variance, clip at 10 SD)
X = adata.X if not hasattr(adata.X, 'toarray') else adata.X.toarray()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = np.clip(X_scaled, -10, 10)  # clip outliers

# Step 2: PCA
n_pcs = min(50, X_scaled.shape[1] - 1)
pca = PCA(n_components=n_pcs, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Store PCA coordinates in adata
adata.obsm['X_pca'] = X_pca
print(f"PCA shape: {X_pca.shape}")
print(f"Variance explained by PC1: {pca.explained_variance_ratio_[0]:.3f} ({pca.explained_variance_ratio_[0]*100:.1f}%)")
print(f"Total variance in top 10 PCs: {pca.explained_variance_ratio_[:10].sum():.3f} ({pca.explained_variance_ratio_[:10].sum()*100:.1f}%)")

# Elbow plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cumvar = np.cumsum(pca.explained_variance_ratio_) * 100
axes[0].plot(range(1, n_pcs+1), pca.explained_variance_ratio_*100, 'o-', ms=4)
axes[0].axvline(15, color='red', linestyle='--', label='Suggested elbow: PC15')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance explained (%)')
axes[0].set_title('Scree plot (variance per PC)')
axes[0].legend()

axes[1].plot(range(1, n_pcs+1), cumvar, 'o-', ms=4, color='orange')
axes[1].axhline(80, color='gray', linestyle='--', label='80% variance')
axes[1].set_xlabel('Number of PCs')
axes[1].set_ylabel('Cumulative variance (%)')
axes[1].set_title('Cumulative variance explained')
axes[1].legend()

plt.tight_layout()
plt.savefig('pca_variance.png', dpi=100, bbox_inches='tight')
plt.show()

# PCA scatter plot colored by true cell type
n_pcs_use = 15
fig, ax = plt.subplots(figsize=(8, 6))
cell_types_list = adata.obs['cell_type'].unique()
colors_map = dict(zip(cell_types_list, ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00']))
for ct in cell_types_list:
    mask = adata.obs['cell_type'] == ct
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors_map[ct], label=ct, s=15, alpha=0.7)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA: PC1 vs PC2 colored by cell type')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('pca_scatter.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"\nUsing top {n_pcs_use} PCs for downstream neighbor graph")

## 2. Neighborhood Graph and UMAP

### Building the k-NN graph
After PCA, we construct a **k-nearest-neighbor graph** in PC space: for each cell, find its k most similar cells (by Euclidean distance in PC space) and connect them with edges. This graph is the foundation for both clustering and UMAP.

Key parameters:
- **`n_neighbors`** (default 15): Controls the local vs global balance. Small values (5–10) emphasize local structure — tight clusters. Large values (30–50) emphasize global structure — better topology but fuzzier clusters. For PBMC-scale data (10k cells), n_neighbors=15–20 is typical.
- **`n_pcs`**: How many PCs to use. Should match your elbow plot analysis.

### UMAP vs t-SNE
Both are non-linear dimensionality reduction methods that project the k-NN graph into 2D.

| | UMAP | t-SNE |
|--|------|-------|
| Speed | Fast (minutes) | Slow (hours for >50k cells) |
| Global structure | Partially preserved | Not preserved |
| Cluster distances | Approximate (meaningful) | Meaningless |
| Reproducibility | Random seed controlled | Less stable |

**Critical warning**: UMAP distances between clusters are NOT direct measures of transcriptional similarity. Two clusters far apart in UMAP space are not necessarily more different than two clusters that are adjacent. For quantitative comparisons, always use the PC space or expression values directly.

### UMAP hyperparameters
- `min_dist` (default 0.5): Controls how tightly cells are packed within clusters. Smaller = tighter clusters, more visual separation.
- `spread` (default 1.0): Controls overall spread of the embedding.
These affect aesthetics but not biological conclusions — don't over-tune them.

In [ ]:
# Build k-NN graph and UMAP
from sklearn.neighbors import NearestNeighbors
import numpy as np
import matplotlib.pyplot as plt

n_pcs_use = 15
n_neighbors = 15

# Build k-NN graph using PCA coordinates
knn = NearestNeighbors(n_neighbors=n_neighbors, metric='euclidean', n_jobs=-1)
knn.fit(adata.obsm['X_pca'][:, :n_pcs_use])
distances, indices = knn.kneighbors(adata.obsm['X_pca'][:, :n_pcs_use])

# Store connectivity in adata (adjacency-like structure)
adata.uns['neighbors'] = {'params': {'n_neighbors': n_neighbors, 'n_pcs': n_pcs_use}}
adata.obsm['knn_indices'] = indices
adata.obsm['knn_distances'] = distances

print(f"k-NN graph: {adata.n_obs} cells x {n_neighbors} neighbors each")
print(f"Mean distance to nearest neighbor: {distances[:, 1].mean():.4f}")

# UMAP
try:
    from umap import UMAP
    reducer = UMAP(n_neighbors=n_neighbors, min_dist=0.3, random_state=42, n_epochs=200)
    X_umap = reducer.fit_transform(adata.obsm['X_pca'][:, :n_pcs_use])
    adata.obsm['X_umap'] = X_umap
    print(f"UMAP embedding shape: {X_umap.shape}")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(8, 6))
    cell_types_list = adata.obs['cell_type'].unique()
    colors_map = dict(zip(sorted(cell_types_list), ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00']))
    for ct in sorted(cell_types_list):
        mask = adata.obs['cell_type'] == ct
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1], c=colors_map[ct], label=ct, s=10, alpha=0.8)
    ax.set_xlabel('UMAP1')
    ax.set_ylabel('UMAP2')
    ax.set_title('UMAP colored by cell type')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('umap_celltypes.png', dpi=100, bbox_inches='tight')
    plt.show()
except ImportError:
    print("umap-learn not installed. Install with: pip install umap-learn")
    print("Falling back to PCA visualization for clustering steps")
    adata.obsm['X_umap'] = adata.obsm['X_pca'][:, :2]

## 3. Leiden / Louvain Clustering

### Graph-based community detection
Clustering in scRNA-seq is not traditional k-means or hierarchical clustering. Instead, **community detection** algorithms find groups of cells that are densely connected in the k-NN graph. These algorithms were originally developed for social network analysis.

**Leiden vs Louvain**: Both maximize modularity (ratio of edges within communities to edges between communities). Leiden is strictly better — it guarantees well-connected communities (Louvain can produce internally disconnected communities). Use Leiden.

### The resolution parameter
Resolution controls how many clusters you get:
- High resolution → more, smaller clusters (over-clustering)
- Low resolution → fewer, larger clusters (under-clustering)

There is no single "correct" resolution. Use domain knowledge:
- Do clusters correspond to known cell types?
- Are marker genes specific to clusters?
- Do adjacent clusters in UMAP space have qualitatively different biology?

**Practical approach**: Start with resolution=0.5, then increase (0.8, 1.0) if you suspect merged populations or decrease (0.3) if you have too much fragmentation.

### Leiden requires the `leidenalg` package
In scanpy: `sc.tl.leiden(adata, resolution=0.5)`. The implementation in scanpy uses the `leidenalg` Python package, which wraps the C++ igraph library for performance.

In [ ]:
# Leiden clustering on the k-NN graph
# We implement a simplified community detection to demonstrate the concept
# In production, use: sc.tl.leiden(adata, resolution=0.5)

import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt

# Build sparse adjacency matrix from k-NN indices
n_cells = adata.n_obs
n_neighbors = adata.obsm['knn_indices'].shape[1]
row_idx = np.repeat(np.arange(n_cells), n_neighbors)
col_idx = adata.obsm['knn_indices'].flatten()
# Mutual k-NN (keep edge only if both cells consider each other neighbors)
adj = sp.csr_matrix((np.ones(len(row_idx)), (row_idx, col_idx)), shape=(n_cells, n_cells))
adj = ((adj + adj.T) > 0).astype(np.float32)  # symmetrize

# Simple connected-components as a proxy for clusters
# (production code uses Leiden/Louvain modularity optimization)
from scipy.sparse.csgraph import connected_components
n_components, labels = connected_components(adj, directed=False)
print(f"Connected components: {n_components}")

# For demonstration, use agglomerative clustering on PCA coordinates
from sklearn.cluster import AgglomerativeClustering
n_clusters = 5
agg = AgglomerativeClustering(n_clusters=n_clusters, metric='euclidean', linkage='ward')
cluster_labels = agg.fit_predict(adata.obsm['X_pca'][:, :15])
adata.obs['leiden'] = [str(c) for c in cluster_labels]  # string labels like Leiden

print(f"\nCluster sizes:")
print(adata.obs['leiden'].value_counts().sort_index())

# Compare clusters to true cell types
from pandas import crosstab
ct = crosstab(adata.obs['leiden'], adata.obs['cell_type'], normalize='index').round(2)
print(f"\nCluster composition (row-normalized):")
print(ct)

# Visualize clusters on UMAP/PCA
coords = adata.obsm['X_umap']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_cluster = ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00']
for ci in sorted(adata.obs['leiden'].unique()):
    mask = adata.obs['leiden'] == ci
    axes[0].scatter(coords[mask, 0], coords[mask, 1], 
                    c=colors_cluster[int(ci)], label=f'Cluster {ci}', s=10, alpha=0.8)
axes[0].set_title('Clustering result')
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left')

cell_types_list = sorted(adata.obs['cell_type'].unique())
colors_map = dict(zip(cell_types_list, ['#e41a1c','#377eb8','#4daf4a','#984ea3','#ff7f00']))
for ct_name in cell_types_list:
    mask = adata.obs['cell_type'] == ct_name
    axes[1].scatter(coords[mask, 0], coords[mask, 1],
                    c=colors_map[ct_name], label=ct_name, s=10, alpha=0.8)
axes[1].set_title('True cell types')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left')

for ax in axes:
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
plt.tight_layout()
plt.savefig('clustering.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Differential Expression Between Clusters

### Finding cluster marker genes
After clustering, the key question is: which genes distinguish each cluster from all others? This is **one-vs-rest** differential expression — for each cluster, compare it against all remaining cells.

**Why Wilcoxon rank-sum test?**
- Non-parametric: makes no distributional assumptions (important because scRNA-seq counts are zero-inflated)
- Computationally efficient
- Interpretable: tests whether the rank ordering of expression differs between groups
- False-positive rate well-controlled in benchmarking studies (Squair et al. 2021)

**What to avoid**: t-tests on log-normalized data inflate false positives because they assume normally distributed residuals. Logistic regression (Seurat) and mixed models (MAST) are alternatives for datasets with batch covariates.

### Key output columns from `sc.tl.rank_genes_groups`
- `names`: gene names, sorted by significance
- `logfoldchanges`: log2 fold-change (cluster vs rest)
- `pvals_adj`: Benjamini-Hochberg corrected p-values
- `scores`: test statistic (z-score for Wilcoxon)

### One-vs-one comparisons
One-vs-rest can miss markers for small clusters (the "rest" averages out many cell types). For distinguishing similar clusters (CD4 vs CD8 T cells), use one-vs-one: `sc.tl.rank_genes_groups(adata, 'leiden', groups=['3'], reference='2')`.

In [ ]:
# Differential expression: find marker genes per cluster
from scipy.stats import ranksums
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

X = adata.X if not hasattr(adata.X, 'toarray') else adata.X.toarray()
clusters = adata.obs['leiden'].values
gene_names = adata.var_names.tolist()

def wilcoxon_de(X, group_mask, n_genes_test=50):
    """One-vs-rest Wilcoxon test for top expressed genes."""
    results = []
    # Test only genes with any expression in the group (speed up)
    group_expr = X[group_mask, :].mean(axis=0)
    top_gene_idx = np.argsort(-group_expr)[:n_genes_test]
    
    for g_idx in top_gene_idx:
        group_vals = X[group_mask, g_idx]
        rest_vals = X[~group_mask, g_idx]
        stat, pval = ranksums(group_vals, rest_vals)
        lfc = np.log2(group_vals.mean() + 1e-9) - np.log2(rest_vals.mean() + 1e-9)
        results.append({'gene': gene_names[g_idx], 'lfc': lfc, 'pval': pval, 'stat': stat})
    
    df = pd.DataFrame(results)
    # Benjamini-Hochberg correction
    from statsmodels.stats.multitest import multipletests
    _, pvals_adj, _, _ = multipletests(df['pval'], method='fdr_bh')
    df['pval_adj'] = pvals_adj
    return df.sort_values('pval_adj')

# Run for each cluster
marker_results = {}
for cluster_id in sorted(adata.obs['leiden'].unique()):
    mask = (clusters == cluster_id)
    de_df = wilcoxon_de(X, mask, n_genes_test=100)
    marker_results[cluster_id] = de_df
    top5 = de_df.head(5)['gene'].tolist()
    print(f"Cluster {cluster_id} (n={mask.sum()}) top markers: {top5}")

# Heatmap of top 3 markers per cluster
top_markers = []
for cid in sorted(marker_results.keys()):
    top_markers.extend(marker_results[cid].head(3)['gene'].tolist())
top_markers = list(dict.fromkeys(top_markers))  # deduplicate, preserve order

# Mean expression per cluster for selected markers
marker_idx = [gene_names.index(g) for g in top_markers if g in gene_names]
cluster_ids_sorted = sorted(adata.obs['leiden'].unique())
heatmap_data = np.zeros((len(cluster_ids_sorted), len(marker_idx)))
for i, cid in enumerate(cluster_ids_sorted):
    mask = clusters == cid
    heatmap_data[i, :] = X[mask][:, marker_idx].mean(axis=0)

fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(heatmap_data, aspect='auto', cmap='RdYlBu_r')
ax.set_xticks(range(len(marker_idx)))
ax.set_xticklabels([gene_names[i] for i in marker_idx], rotation=90, fontsize=7)
ax.set_yticks(range(len(cluster_ids_sorted)))
ax.set_yticklabels([f'Cluster {c}' for c in cluster_ids_sorted])
ax.set_title('Mean expression of top cluster markers')
plt.colorbar(im, ax=ax, label='Mean log-norm expression')
plt.tight_layout()
plt.savefig('marker_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. Visualizing Marker Genes

### Dot plot
The dot plot is the most information-dense single-cell visualization: each dot shows two variables simultaneously:
- **Dot size**: fraction of cells in the cluster expressing the gene (> 0)
- **Dot color**: mean expression level among expressing cells

This makes it easy to distinguish a marker that is expressed at medium level in all cells vs one expressed at high level in 30% of cells.

### Feature plot (gene expression on UMAP)
Overlaying continuous gene expression on UMAP reveals the spatial distribution of marker genes. Best used for known canonical markers to validate cluster identity. Do not use to "discover" markers — it is subject to confirmation bias.

### When clusters don't match biology
If your clustering produces clusters that don't have clear marker gene signatures, consider:
1. Over-clustering: reduce resolution and re-run Leiden
2. Batch effect: two conditions are separating on batch rather than biology — run batch correction (Notebook 3 of Module 31)
3. Insufficient QC: doublets present (each doublet cluster will have two cell types' markers simultaneously)

In [ ]:
# Visualize marker genes
import matplotlib.pyplot as plt
import numpy as np

X = adata.X if not hasattr(adata.X, 'toarray') else adata.X.toarray()
gene_names = adata.var_names.tolist()
clusters = adata.obs['leiden'].values
cluster_ids = sorted(adata.obs['leiden'].unique())
coords = adata.obsm['X_umap']

# Select demonstration marker genes (top marker per cluster)
demo_markers = [marker_results[cid].iloc[0]['gene'] for cid in cluster_ids]
print(f"Top marker per cluster: {list(zip(cluster_ids, demo_markers))}")

# --- Dot plot ---
frac_exp = np.zeros((len(cluster_ids), len(demo_markers)))
mean_exp = np.zeros_like(frac_exp)
for i, cid in enumerate(cluster_ids):
    mask = clusters == cid
    for j, gene in enumerate(demo_markers):
        if gene in gene_names:
            g_idx = gene_names.index(gene)
            vals = X[mask, g_idx]
            frac_exp[i, j] = (vals > 0).mean()
            mean_exp[i, j] = vals[vals > 0].mean() if (vals > 0).any() else 0

fig, ax = plt.subplots(figsize=(10, 4))
for i in range(len(cluster_ids)):
    for j in range(len(demo_markers)):
        size = frac_exp[i, j] * 400  # scale dot size
        color_val = mean_exp[i, j]
        ax.scatter(j, i, s=size, c=[color_val], cmap='Reds', vmin=0,
                   vmax=mean_exp.max(), edgecolors='gray', linewidth=0.5)

ax.set_xticks(range(len(demo_markers)))
ax.set_xticklabels(demo_markers, rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(cluster_ids)))
ax.set_yticklabels([f'Cluster {c}' for c in cluster_ids])
ax.set_title('Dot plot: top marker per cluster')
ax.set_xlabel('Marker gene')
sm = plt.cm.ScalarMappable(cmap='Reds', norm=plt.Normalize(0, mean_exp.max()))
plt.colorbar(sm, ax=ax, label='Mean expression')
plt.tight_layout()
plt.savefig('dotplot.png', dpi=100, bbox_inches='tight')
plt.show()

# --- Feature plots: gene expression on UMAP ---
n_genes_plot = min(4, len(demo_markers))
fig, axes = plt.subplots(1, n_genes_plot, figsize=(4*n_genes_plot, 4))
if n_genes_plot == 1:
    axes = [axes]
for ax, gene in zip(axes, demo_markers[:n_genes_plot]):
    if gene in gene_names:
        g_idx = gene_names.index(gene)
        expr = X[:, g_idx]
        sc = ax.scatter(coords[:, 0], coords[:, 1], c=expr, cmap='YlOrRd',
                       s=8, alpha=0.8)
        ax.set_title(gene, fontsize=10)
        ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
        plt.colorbar(sc, ax=ax, shrink=0.7)
plt.suptitle('Feature plots: marker gene expression', y=1.02)
plt.tight_layout()
plt.savefig('feature_plots.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nNext steps: Module 30 Notebook 3 — annotate clusters with known cell type markers")

---[< Previous: scRNA-seq: Quality Control and Preprocessing](01_scrna_preprocessing.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: scRNA-seq: Cell Type Annotation >](03_cell_type_annotation.ipynb)---